# 🚀 Mastering AI Agents: Multi-Agent Orchestration with Google ADK! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

Labs and Goals:
- **Lab2: Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Lab 3: Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Lab 4: Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **[HERE]>>>Demo: Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

Credit: Notebook content adapted from Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, who passionate about helping developers build with AI and cloud technologies.



-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: [Google AI Studio](https://aistudio.google.com/app/apikey)

 -----------------------------------------------------------------------------


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [ ]:
# --- Ying Cell ID 1----#
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")



✅ All libraries are ready to go!


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


### Configure Your API Key
To use Gemini models, you need an API key from [Google AI Studio](https://aistudio.google.com/app/apikey). This section securely collects your key and configures it for the ADK.


In [ ]:
# --- Ying Cell ID 2----#
# --- API Key Configuration ---
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the 🔑 icon in the left sidebar, add a secret named GOOGLE_API_KEY
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    # Option 2: Paste it directly (less secure but fine for learning)
    import getpass
    GOOGLE_API_KEY = getpass.getpass("🔑 Enter your Google AI Studio API key: ")
    print("✅ API key entered manually.")


✅ API key loaded from Colab Secrets.


In [ ]:
# --- Ying Cell ID 3----#
# --- Set Environment Variables for ADK ---

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

print(f"✅ API key configured (starts with '{GOOGLE_API_KEY[:6]}...')")
print("✅ Using Google AI Studio (not Vertex AI).")


✅ API key configured (starts with 'AQ.Ab8...')
✅ Using Google AI Studio (not Vertex AI).


In [ ]:
# --- Ying Cell ID 6 ----#

# For Part2: You also have to run This Cell together with Cell 1 and Cell 2

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [ ]:
# --- Ying Cell ID 5 ----# Share with Scenario 3

# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [ ]:
# --- Ying Cell ID 13 ----#

# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.
        **Always respond in English, regardless of the destination's local language.**        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario A: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [ ]:
# --- Ying Cell ID 14 ----# require run_agent_query in Cell 5 also

# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: d7e52b99-d9b3-4db6-9caa-47ec4b1c2ba7

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'd7e52b99-d9b3-4db6-9caa-47ec4b1c2ba7'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Hello! A 2-day trip to Lisbon with a focus on historic sites and great local food sounds fantastic! I'll help you plan a memorable itinerary.

Let's start with **Day 1**:

### Day 1: Historic Alfama & Downtown Delights

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & São Jorge Castle**
    Begin your day in Alfama, Lisbon's oldest and most picturesque neighborhood, known for its narrow, winding streets and historic charm.
    *   Wander through its labyrinthine alleys, soak in the atmosphere, and discover hidden viewpoints like M

Hello! A 2-day trip to Lisbon with a focus on historic sites and great local food sounds fantastic! I'll help you plan a memorable itinerary.

Let's start with **Day 1**:

### Day 1: Historic Alfama & Downtown Delights

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & São Jorge Castle**
    Begin your day in Alfama, Lisbon's oldest and most picturesque neighborhood, known for its narrow, winding streets and historic charm.
    *   Wander through its labyrinthine alleys, soak in the atmosphere, and discover hidden viewpoints like Miradouro de Santa Luzia and Miradouro das Portas do Sol, offering stunning panoramic views of the city and the Tagus River.
    *   Visit the Lisbon Cathedral (Sé de Lisboa), a beautiful medieval cathedral with over 800 years of history.
    *   Continue your journey up to São Jorge Castle (Castelo de São Jorge), an ancient Moorish fortress offering breathtaking 360-degree views over Lisbon.

*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Flavors**
    Head to a local "tasca" (traditional tavern) in Alfama or the Baixa district to savor some authentic Portuguese cuisine. Try a "Bifana" (pork sandwich), a beloved street food, or "Bacalhau à Brás" (shredded cod with potatoes and eggs).

*   **Afternoon (2:30 PM - 6:00 PM): Baixa, Rossio Square & Elevador de Santa Justa**
    *   Descend into the Baixa district, the heart of Lisbon, characterized by its grand squares and traditional shops. Explore Rossio Square, a classic meeting point with its distinctive wave-patterned paving.
    *   Stroll along Rua Augusta, a lively commercial street, and walk under the iconic Rua Augusta Arch, a symbol of resilience after the 1755 earthquake.
    *   Consider a ride on the Elevador de Santa Justa for unique views over the city (though expect queues).

*   **Evening (6:00 PM onwards): Time Out Market & Ginjinha**
    *   For dinner, head to the Time Out Market Lisboa. This large food hall brings together some of the city's best-known chefs and food stalls under one roof, offering a wide variety of Portuguese dishes.
    *   After dinner, try a shot of Ginjinha, Lisbon's traditional sour cherry liqueur, often served in a chocolate cup. You can find local spots selling it in Alfama or near Rossio Square.

How does this sound for your first day in Lisbon?

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'd7e52b99-d9b3-4db6-9caa-47ec4b1c2ba7'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, no problem at all! I understand that not everyone enjoys castles. Let's swap out São Jorge Castle for another fascinating historical site in Lisbon.

Here's the revised plan for **Day 1**:

### Day 1: Historic Alfama & Downtown Delights

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & National Pantheon**
    Begin your day in Alfama, Lisbon's oldest and most picturesque neighborhood, known for its narrow, winding streets and historic charm.
    *   Wander through its labyrinthine alleys, soak in the atmosphere, and discover hidden viewpoints like Miradouro

Okay, no problem at all! I understand that not everyone enjoys castles. Let's swap out São Jorge Castle for another fascinating historical site in Lisbon.

Here's the revised plan for **Day 1**:

### Day 1: Historic Alfama & Downtown Delights

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & National Pantheon**
    Begin your day in Alfama, Lisbon's oldest and most picturesque neighborhood, known for its narrow, winding streets and historic charm.
    *   Wander through its labyrinthine alleys, soak in the atmosphere, and discover hidden viewpoints like Miradouro de Santa Luzia and Miradouro das Portas do Sol, offering stunning panoramic views of the city and the Tagus River.
    *   Visit the Lisbon Cathedral (Sé de Lisboa), a beautiful medieval cathedral with over 800 years of history.
    *   Instead of the castle, we'll head to the **National Pantheon (Panteão Nacional)**. This magnificent 17th-century building, originally the Church of Santa Engrácia, houses the tombs of some of Portugal's most important historical figures. Its impressive dome also offers fantastic views of the city and the Tagus River.

*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Flavors**
    Head to a local "tasca" (traditional tavern) in Alfama or the Baixa district to savor some authentic Portuguese cuisine. Try a "Bifana" (pork sandwich), a beloved street food, or "Bacalhau à Brás" (shredded cod with potatoes and eggs).

*   **Afternoon (2:30 PM - 6:00 PM): Baixa, Rossio Square & Elevador de Santa Justa**
    *   Descend into the Baixa district, the heart of Lisbon, characterized by its grand squares and traditional shops. Explore Rossio Square, a classic meeting point with its distinctive wave-patterned paving.
    *   Stroll along Rua Augusta, a lively commercial street, and walk under the iconic Rua Augusta Arch, a symbol of resilience after the 1755 earthquake.
    *   Consider a ride on the Elevador de Santa Justa for unique views over the city (though expect queues).

*   **Evening (6:00 PM onwards): Time Out Market & Ginjinha**
    *   For dinner, head to the Time Out Market Lisboa. This large food hall brings together some of the city's best-known chefs and food stalls under one roof, offering a wide variety of Portuguese dishes.
    *   After dinner, try a shot of Ginjinha, Lisbon's traditional sour cherry liqueur, often served in a chocolate cup. You can find local spots selling it in Alfama or near Rossio Square.

How does this revised Day 1 plan sound to you?

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'd7e52b99-d9b3-4db6-9caa-47ec4b1c2ba7'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Excellent! I'm glad Day 1 is perfect for you.

Let's move on to **Day 2**, where we'll explore more of Lisbon's rich history and, of course, indulge in more fantastic local food!

### Day 2: Belém's Maritime History & Culinary Delights

*   **Morning (9:00 AM - 1:00 PM): Belém's Age of Discoveries**
    *   Start your day in the historic Belém district, significant for Portugal's Age of Discoveries.
    *   Visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site and an outstanding example of Manueline architecture. Explore its stunning cloisters and the church, where th

Excellent! I'm glad Day 1 is perfect for you.

Let's move on to **Day 2**, where we'll explore more of Lisbon's rich history and, of course, indulge in more fantastic local food!

### Day 2: Belém's Maritime History & Culinary Delights

*   **Morning (9:00 AM - 1:00 PM): Belém's Age of Discoveries**
    *   Start your day in the historic Belém district, significant for Portugal's Age of Discoveries.
    *   Visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site and an outstanding example of Manueline architecture. Explore its stunning cloisters and the church, where the explorer Vasco da Gama is buried.
    *   Walk along the Tagus River to the iconic **Belém Tower (Torre de Belém)**, another UNESCO site. This fortified tower served as a ceremonial gateway to Lisbon and is a symbol of Portugal's maritime power.
    *   See the impressive **Monument to the Discoveries (Padrão dos Descobrimentos)**, which celebrates the Portuguese explorers and their role in the Age of Discovery.

*   **Lunch/Snack (1:00 PM - 2:00 PM): The Original Pastéis de Belém**
    *   No trip to Belém is complete without a visit to the famous "Pastéis de Belém" bakery. Indulge in their warm, freshly baked, and iconic custard tarts, a truly unforgettable Lisbon culinary experience.

*   **Afternoon (2:00 PM - 5:30 PM): Elegant Chiado & Bohemian Bairro Alto**
    *   Take a tram or bus back towards the central areas.
    *   Explore **Chiado**, an elegant and historic neighborhood known for its beautiful theaters, traditional cafés, and charming bookstores.
    *   Wander into **Bairro Alto**, a district with a bohemian vibe during the day, offering incredible views. Head to **Miradouro de São Pedro de Alcântara** for another breathtaking panoramic vista of the city, looking back towards Day 1's Alfama and the castle.

*   **Evening (6:00 PM onwards): Fado Show & Traditional Dinner**
    *   For your final evening, immerse yourself in Portuguese culture by experiencing a traditional **Fado show** paired with a delicious dinner. Fado, a soulful music genre, is a UNESCO Intangible Cultural Heritage and often performed in intimate restaurants in areas like Alfama or Bairro Alto. This combines authentic local food with a unique historical and emotional cultural experience.

How does this plan for Day 2 sound for concluding your Lisbon trip?

--------------------------------------------------



### Scenario B: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [ ]:
# --- Ying Cell ID 15 ----#

# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: 72f469e4-9fbc-4e33-baca-1d82d8ae6a84
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '72f469e4-9fbc-4e33-baca-1d82d8ae6a84'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Hello there! I'd love to help you plan your 2-day trip to Lisbon, Portugal, focusing on historic sites and delicious local food.

Let's start with **Day 1**. How about this itinerary?

### Day 1: Historic Alfama & Baixa

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day by getting lost in the narrow, winding streets of Alfama, Lisbon's oldest district.
    *   Visit **São Jorge Castle (Cast

Hello there! I'd love to help you plan your 2-day trip to Lisbon, Portugal, focusing on historic sites and delicious local food.

Let's start with **Day 1**. How about this itinerary?

### Day 1: Historic Alfama & Baixa

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day by getting lost in the narrow, winding streets of Alfama, Lisbon's oldest district.
    *   Visit **São Jorge Castle (Castelo de São Jorge)** for panoramic views of the city and the Tagus River.
    *   Wander past the **Lisbon Cathedral (Sé de Lisboa)**, a beautiful Romanesque cathedral.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Cuisine**
    *   Enjoy a traditional Portuguese lunch in Alfama. I can suggest some local tascas (small, traditional restaurants) if you'd like!
*   **Afternoon (2:30 PM - 6:00 PM): Baixa District & Commerce Square**
    *   Head to the Baixa district, rebuilt after the 1755 earthquake, known for its neoclassical architecture and grid-like streets.
    *   Walk through **Praça do Comércio (Commerce Square)**, one of Europe's largest and most magnificent squares, right on the riverfront.
    *   Take a ride on the **Santa Justa Lift** for more great views (though be prepared for queues!).
*   **Evening (7:00 PM onwards): Dinner & Fado Show**
    *   Savor a delicious dinner in the Baixa or Chiado area.
    *   Experience a traditional **Fado show**, a soulful Portuguese musical genre, often accompanied by dinner in Alfama or Bairro Alto.

How does this sound for your first day in Lisbon? Are there any changes you'd like to make, or shall we move on to plan Day 2?

--------------------------------------------------


Created a BRAND NEW session for Turn 2: 6d77f769-40fc-41e9-b663-32b096f853dc
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '6d77f769-40fc-41e9-b663-32b096f853dc'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, great! I'm glad Day 1 looks good.

To plan Day 2, could you please remind me of our destination, the total duration of your trip, and your interests? This will help me ensure the activities for Day 2 build logically and uniquely on what we've already planned."""
    ),
  ],
  role='model'
) grounding_metadata=GroundingMetadata() partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=64,
  prompt_token_count=330,
  prompt_tokens_de

Okay, great! I'm glad Day 1 looks good.

To plan Day 2, could you please remind me of our destination, the total duration of your trip, and your interests? This will help me ensure the activities for Day 2 build logically and uniquely on what we've already planned.

--------------------------------------------------



See? The agent was confused! It likely asked what destination

or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
